# project_09_macrocycle_mdm2 — all notebooks (00→05) in one

This is a **convenience copy** that concatenates the six standalone notebooks in order so you can run the whole project top-to-bottom in a single Colab session. The individual notebooks (`00_setup.ipynb` … `05_validation_plan.ipynb`) remain in this folder and are the canonical deliverables. Sections are separated by dividers; each section keeps its own setup/`import` cells (re-running them is harmless). All synthetic numbers are still labeled `EXAMPLE_DATA`.

---

## ▶︎ Section 1 / 6 — `00_setup.ipynb`

---

# 00 · Environment Setup — De Novo Protein Design Capstone

This is the **shared setup notebook** every project starts from. Run it top to bottom
*once per Colab session*. It:

1. detects your GPU and warns if you're on a weak/absent one,
2. installs a light, pinned core toolset (Biopython, py3Dmol, foldseek-less utilities),
3. optionally installs heavier tools (ColabFold, ESMFold) on demand,
4. prints exact versions for your `LOG.md` (reproducibility is graded).

> **Compute reality.** A free Colab **T4** runs ColabFold, ESMFold, ProteinMPNN, and small
> RFdiffusion jobs. **BindCraft / RFantibody / large RFdiffusion** want an **A100** (Colab Pro+
> or a cluster). Each project's `MANUAL.md` states its tier. Don't fight a T4 to do an A100 job —
> plan your batch sizes around it.

## 1 · GPU & environment check

In [ ]:
import subprocess, sys, platform, textwrap

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

print("Python :", sys.version.split()[0])
print("Platform:", platform.platform())

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null")
if gpu:
    print("GPU    :", gpu)
    name = gpu.lower()
    if "t4" in name:
        print(textwrap.fill(
            "NOTE: T4 detected. Good for ColabFold/ESMFold/ProteinMPNN/small RFdiffusion. "
            "For BindCraft/RFantibody/large diffusion, switch to A100 (Colab Pro+) or a cluster.", 88))
    elif any(x in name for x in ("a100", "l4", "v100")):
        print("NOTE: capable GPU — heavier tools (BindCraft/RFantibody) are feasible.")
else:
    print("GPU    : NONE FOUND")
    print(textwrap.fill(
        "WARNING: No GPU. Go to Runtime → Change runtime type → Hardware accelerator → GPU. "
        "Structure prediction on CPU is impractically slow.", 88))

## 2 · Pinned core install (fast, T4-friendly)

These are light and used across every project. Pins are conservative; bump them in your repo if needed and **log it**.

In [ ]:
# Core utilities used in every project. Quiet + pinned.
%pip -q install biopython==1.84 py3Dmol==2.4.0 numpy pandas matplotlib seaborn tqdm requests 2>/dev/null
print("Core install done.")

In [ ]:
# Version stamp — copy this block's output into your LOG.md for reproducibility.
import importlib, datetime
mods = ["Bio", "py3Dmol", "numpy", "pandas", "matplotlib", "seaborn", "tqdm", "requests"]
print("# Environment stamp", datetime.datetime.utcnow().isoformat(timespec="seconds"), "UTC")
for m in mods:
    try:
        v = importlib.import_module(m).__version__
    except Exception:
        v = "n/a"
    print(f"{m:14s} {v}")

## 3 · Heavy tools — install *on demand*

Don't install these unless your project needs them this session (they're slow to set up).
Each is wrapped in a function so you only pay the cost when you call it.

In [ ]:
def install_colabfold():
    """ColabFold (AF2). ~3–5 min on first install. T4 OK."""
    import subprocess
    subprocess.run("pip -q install 'colabfold[alphafold-minus-jax]'", shell=True)
    # On Colab, the standard route is the localcolabfold installer or the ColabFold notebook;
    # here we expose the pip route. If it fails, fall back to the official ColabFold notebook
    # and import your sequences. Log whichever path you used.
    print("ColabFold install attempted. Verify with: from colabfold.batch import run")

def install_esmfold():
    """ESMFold via HuggingFace transformers. T4 OK for <~400 aa."""
    import subprocess
    subprocess.run("pip -q install 'transformers>=4.40' accelerate", shell=True)
    print("ESMFold deps installed. Load with transformers EsmForProteinFolding.")

print("Helpers ready: install_colabfold(), install_esmfold().")

## 4 · Reproducibility helpers

Call `set_seeds()` at the top of every run, and use `log()` to append to your `LOG.md`.

In [ ]:
import os, random
import numpy as np

def set_seeds(seed: int = 0):
    random.seed(seed); np.random.seed(seed); os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass
    print(f"seeds set to {seed}")

def log(msg: str, path: str = "LOG.md"):
    import datetime
    stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
    with open(path, "a") as fh:
        fh.write(f"- {stamp}Z · {msg}\n")
    print("logged:", msg)

set_seeds(0)
log("Ran 00_setup; environment stamped.")

## 5 · (Optional) Mount Google Drive for persistence

Colab sessions are ephemeral. Mount Drive to keep your `results/` and design pools between sessions.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# WORKDIR = "/content/drive/MyDrive/denovo_capstone/project_XX"
# import os; os.makedirs(WORKDIR, exist_ok=True); os.chdir(WORKDIR)
print("Uncomment to mount Drive and set your working directory.")

---
**Next:** open `01_define_and_explore.ipynb`. Keep this session alive — re-running `00_setup`
each new session is normal. Record every version and seed in `LOG.md`.

---

## ▶︎ Section 2 / 6 — `01_define_and_explore.ipynb`

---

# 01 · Define & Explore — MDM2 cleft prep, p53 sub-pockets, peptide metrics

**Standard slot:** *define & explore.* **For Project 09 this means:** clean the MDM2 N-terminal
domain, define the **p53-binding cleft** (the Phe19/Trp23/Leu26 sub-pockets) as the design site, write
down the binder metrics + cutoffs, and run a deterministic **mock** mini-run (a few linear + cyclic
peptides) as your "hello-world" (D0).

Run `00_setup.ipynb` first in this session. Peptides are small, so AF2/Boltz-2 scoring and Boltz-2
affinity on small inputs run on a **free T4**; a full macrocycle campaign prefers **Colab Pro** (see
`MANUAL.md §2`). Everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab.

## The peptide / macrocycle metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the peptide | thermostability / protease stability / permeability |
| **pae_interaction** | Å | AF2/Boltz-2 error across the **peptide–MDM2 interface** (the key metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency; noisier for short peptides) | binding/function |
| **boltz_affinity_score** | rel. | Boltz-2 **relative** affinity ranking signal | **a K_D** — never |
| shape complementarity | 0–1 | interface packing quality in the cleft | epitope correctness |
| cleft overlap | 0–1 | fraction of the p53 sub-pockets the peptide covers | a guarantee it displaces p53 |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** `pae_interaction` is the single most important interface metric — but a low value is
*confidence*, **not** affinity. The **Boltz-2 affinity score is a relative RANK, not a K_D.** A passing
design is a **hypothesis** until synthesis + binding/stability assays — and **predicted affinity for
short peptides is unreliable: rank, don't trust.**

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Target prep + the p53 cleft

The design target is the **MDM2 N-terminal domain**, and the "hotspots" are the MDM2 residues lining
the **p53-binding cleft** — the three sub-pockets that bury p53 **Phe19 / Trp23 / Leu26**. Steering a
peptide there is what makes it a *p53-mimetic competitor*. Fetch the candidate complex with
`data/download_data.py` (**1YCR — verify on RCSB**), isolate the MDM2 chain, remove the p53
peptide/waters/heteroatoms, and read the cleft residues off the complex.

Below we just *declare* an EXAMPLE cleft set so the notebook runs end-to-end; **replace it with the
residues you derive from the actual MDM2–p53 interface** (numbering depends on the PDB you verify).

In [ ]:
import peptide_tools as pt

TARGET = "MDM2"                     # cleaned MDM2 N-terminal domain (you produce this from 1YCR)
# EXAMPLE cleft residues lining the p53 sub-pockets — VERIFY/REPLACE from the MDM2-p53 interface (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
CLEFT = pt.parse_cleft("A54,A67,A73,A93,A100")   # EXAMPLE_DATA placeholder residues
print("target:", TARGET)
print("cleft :", CLEFT, " (EXAMPLE — replace with your verified p53-cleft residues from 1YCR)")

## 2 · Mock hello-world: a tiny linear + macrocyclic mini-run

`scripts/peptide_tools.py` exposes the design API: `design_peptide(..., cyclic=False|True)` for linear
and macrocyclic peptides, plus `boltz_affinity(...)` (the relative-ranking scorer). The **mock**
backend is deterministic and GPU-free so you can develop the plumbing. **Never report mock numbers as
real** — they are `SYNTHETIC` by construction, and the affinity score is a **relative rank, not a K_D**.

In [ ]:
# A few linear and a few macrocyclic designs, scored by mock AF2/Boltz-2. All numbers are SYNTHETIC.
lin = pt.design_peptide(TARGET, CLEFT, length=12, cyclic=False, n=3, tool="mock")
cyc = pt.design_peptide(TARGET, CLEFT, length=12, cyclic=True,  n=3, tool="mock")
pt.score_designs(lin, tool="mock")
pt.score_designs(cyc, tool="mock")

d = cyc[0]
print("example MACROCYCLE design:")
print("  id    :", d.design_id)
print("  len   :", d.length, "aa   cyclic:", d.cyclic)
print("  seq   :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd, " sc =", d.shape_complementarity)
print("  boltz_affinity_score =", d.boltz_affinity_score, " (RELATIVE RANK, NOT a K_D; SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'boltzgen'/'evobind2'/'boltz' on Colab. See MANUAL.md §2.")

## 3 · Cleft-engagement proxy (does it cover the p53 sub-pockets?)

A peptide only *competes with p53* if it covers the p53 sub-pockets (Phe19/Trp23/Leu26). `cleft_overlap()`
is a geometry proxy (fraction of cleft residues contacted) — a teaching stand-in for the
p53-displacement assay in notebook 04. Higher ⇒ more likely to displace p53 (not a guarantee of
functional reactivation).

In [ ]:
for b in cyc[:3]:
    ov = pt.cleft_overlap(b.contact_residues, CLEFT)
    print(f"{b.design_id}: contacts {b.contact_residues} -> p53-cleft overlap = {ov} (SYNTHETIC)")

## Visualize a peptide–MDM2 cleft complex (py3Dmol)

Use this to eyeball a predicted peptide–MDM2 complex once you have a real PDB (from AF2/Boltz-2).

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.zoomTo()
    return view.show()

# Example (after a real AF2/Boltz-2 prediction writes a complex PDB):
# show_complex("results/boltz/top_complex.pdb")
print("show_complex(pdb_path) ready.")

## D0 checklist
- [ ] MDM2–p53 accession verified on RCSB (**1YCR** is a candidate); MDM2 chain + N-terminal domain identified.
- [ ] Cleaned MDM2 target + **p53-cleft residue list** (the Phe19/Trp23/Leu26 sub-pockets; derived from the interface, not invented).
- [ ] One-paragraph definition of each metric **with** its "does not mean" note (esp. Boltz-2 score ≠ K_D).
- [ ] Reproduced mock mini-run (linear + cyclic) with metrics printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the linear + macrocyclic peptide campaign.

---

## ▶︎ Section 3 / 6 — `02_generate.ipynb`

---

# 02 · Campaign — linear + macrocyclic peptide design vs MDM2

**Standard slot:** *design campaign.* **For Project 09 this is the core:** run **both** peptide
modalities against the p53 cleft and assemble their pools (D2):
- **Linear peptides** (BoltzGen peptide-anything / EvoBind2) — vary **length**.
- **Macrocycles** (BoltzGen macrocycle-anything / EvoBind2 cyclic) — vary **length + cyclization**.

Then score every design with **AF2/Boltz-2** (`pae_interaction` is the key interface metric) and add
the **Boltz-2 affinity score (relative ranking only — NOT a K_D)**.

> **Compute honesty:** peptides are small — AF2/Boltz-2 scoring and Boltz-2 affinity on small inputs
> run on a **free T4**. A **full macrocycle campaign prefers Colab Pro**. The cells below run on the
> deterministic **mock** backend so the plumbing executes anywhere; the real calls + compute notes are
> shown alongside. Run `00_setup.ipynb` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Version-verify the pinned upstreams (tools change — BoltzGen especially!)

The peptide tools live in fast-moving upstream repos, and the **BoltzGen public release is moving** —
**verify it explicitly**. **Pin commits/releases** and **verify the URLs still exist** before relying
on them (`requests.head`; a non-200 means it moved — update the pin and log it). The check needs no GPU.

In [ ]:
import requests

# Pinned upstreams (pin a COMMIT/tag/release in your repo — these change; record the exact pin):
#   BoltzGen   <VERIFY the current public release URL>   # peptide-anything / macrocycle-anything — MOVING; verify + pin
#   EvoBind2   https://github.com/patrickbryant1/EvoBind  # cyclic/linear peptide binder design; pin <commit>
#   Boltz      https://github.com/jwohlwend/boltz          # Boltz-2 structure + affinity; pin <version>
#   ColabFold  https://github.com/sokrypton/ColabFold      # AF2 / AF2-Multimer pAE; pin <commit>
PINNED = {
    # BoltzGen: replace with the verified current public release URL once confirmed (see MANUAL.md §2).
    "BoltzGen_repo_candidate": "https://github.com/jwohlwend/boltz",  # PLACEHOLDER — VERIFY the real BoltzGen release URL
    "EvoBind2":  "https://github.com/patrickbryant1/EvoBind",
    "Boltz":     "https://github.com/jwohlwend/boltz",
    "ColabFold": "https://github.com/sokrypton/ColabFold",
}
for name, url in PINNED.items():
    try:
        r = requests.head(url, allow_redirects=True, timeout=15)
        print(f"  [{r.status_code}] {name:24s} {url}")
    except Exception as e:  # noqa: BLE001
        print(f"  [ERR] {name:24s} {url}  ({e})")
print("\nNon-200 / error => the upstream moved; update the pin in env/requirements.txt and log it.")
print("BoltzGen especially: VERIFY the current public release URL before relying on it (it is moving).")

## 1 · Define the campaign

Same target + cleft as notebook 01. Set honest campaign sizes and the length/cyclization sweep; the
cells run on `mock` so they execute anywhere. On Colab switch `TOOL_*` to the real backends — and the
**macrocycle arm prefers Pro** (shrink it on a T4).

In [ ]:
import peptide_tools as pt
import pandas as pd

TARGET = "MDM2"
CLEFT = pt.parse_cleft("A54,A67,A73,A93,A100")   # EXAMPLE — replace with your verified p53-cleft residues

# Length / cyclization sweep (catalog: vary length + constraint). Small mock counts here for a fast dry
# run; scale up with the real backend (a few hundred per modality on Colab; macrocycle arm on Pro).
LINEAR_LENGTHS = [8, 12, 16, 20]     # linear peptide lengths to sweep
CYCLIC_LENGTHS = [8, 11, 14]         # macrocycle lengths to sweep
N_PER_LENGTH   = 15                  # designs per length bucket (mock); scale up on Colab

TOOL_DESIGN = "mock"   # -> "boltzgen" / "evobind2" on Colab
TOOL_SCORE  = "mock"   # -> "boltz" (Boltz-2: pAE + relative affinity) / "af2" (ColabFold pAE) on Colab

print(f"linear lengths   : {LINEAR_LENGTHS}")
print(f"cyclic lengths   : {CYCLIC_LENGTHS}")
print(f"n per length     : {N_PER_LENGTH}   design tool={TOOL_DESIGN}  score tool={TOOL_SCORE}")
print("cleft            :", CLEFT)

## 2 · Modality #1 — linear peptide campaign (length sweep)

Linear peptides across the length sweep. On Colab this is BoltzGen peptide-anything and/or EvoBind2;
we re-score with AF2/Boltz-2 so the modality comparison is apples-to-apples. The `mock` backend returns
deterministic `SYNTHETIC` designs.

In [ ]:
# Real call (Colab): pt.design_peptide(TARGET, CLEFT, length=L, cyclic=False, n=..., tool="boltzgen")
#   or tool="evobind2". See MANUAL.md §2 / scripts/peptide_tools.py TODOs.
linear = []
for L in LINEAR_LENGTHS:
    batch = pt.design_peptide(TARGET, CLEFT, length=L, cyclic=False, n=N_PER_LENGTH, tool=TOOL_DESIGN)
    linear.extend(batch)
pt.score_designs(linear, tool=TOOL_SCORE)     # AF2/Boltz-2 -> pae_interaction, plddt, scrmsd, sc, boltz rank
print(f"linear pool: {len(linear)} designs across lengths {LINEAR_LENGTHS} (tool={TOOL_DESIGN}; SYNTHETIC if mock)")
print("example:", linear[0].design_id, "len=", linear[0].length, "pae=", linear[0].pae_interaction)

## 3 · Modality #2 — macrocyclic campaign (length + cyclization sweep)

Macrocycles across the length sweep with the **cyclic** constraint (head-to-tail / side-chain). On
Colab this is BoltzGen macrocycle-anything / EvoBind2 cyclic — and **prefers Colab Pro**. The `mock`
backend stands in for the whole chain.

In [ ]:
# Real call (Colab, Pro for the campaign): pt.design_peptide(TARGET, CLEFT, length=L, cyclic=True,
#   n=..., tool="boltzgen") or tool="evobind2". The macrocycle arm is the heavier one.
macro = []
for L in CYCLIC_LENGTHS:
    batch = pt.design_peptide(TARGET, CLEFT, length=L, cyclic=True, n=N_PER_LENGTH, tool=TOOL_DESIGN)
    macro.extend(batch)
pt.score_designs(macro, tool=TOOL_SCORE)
print(f"macrocycle pool: {len(macro)} designs across lengths {CYCLIC_LENGTHS} (tool={TOOL_DESIGN}; SYNTHETIC if mock)")
print("example:", macro[0].design_id, "len=", macro[0].length, "cyclic=", macro[0].cyclic,
      "pae=", macro[0].pae_interaction)

## 4 · Mini-protein FOIL (for the peptide-vs-protein modality comparison)

To compare *modalities* (notebook 04), we also generate a **mini-protein binder foil** against the
same cleft — a Project-06-style mini-binder. Here it is mocked; on Colab the real path defers to the
Project-06 binder workflow (BindCraft / RFdiffusion-binder + ProteinMPNN, **A100**). Generate it at a
comparable scale so the comparison is fair.

In [ ]:
# Real path (Colab, A100): the Project-06 binder workflow against the same MDM2 cleft.
foil = pt.design_miniprotein_foil(TARGET, CLEFT, n=2 * N_PER_LENGTH, tool="mock")
pt.score_designs(foil, tool=TOOL_SCORE)
print(f"mini-protein foil: {len(foil)} designs (mock; real = Project-06 workflow on A100)")
print("example:", foil[0].design_id, "len=", foil[0].length, "pae=", foil[0].pae_interaction)

## 5 · Assemble + persist the pools

Write one tidy CSV per modality (plus a combined one). These feed notebook 03 (the shared filter).
We add an EXAMPLE physics column (`rosetta_dG`) here so the binder physics layer has something to act
on in the dry run — on Colab replace with a real interface-energy estimate; for `mock` it is SYNTHETIC.

In [ ]:
import pandas as pd

def pool_to_df(designs):
    rows = []
    for d in designs:
        # In the mock dry run we attach an EXAMPLE_DATA interface energy so Layer 3 (physics) is
        # exercised. On Colab, replace with a real interface-energy / solubility estimate.
        rdg = -45.0 + (pt._hashints("dG", d.design_id) % 40)   # SYNTHETIC, range ~ -45..-6 REU
        rows.append(dict(
            design_id=d.design_id, modality=d.modality, tool=d.tool, target=d.target,
            length=d.length, cyclic=d.cyclic, sequence=d.sequence,
            plddt=d.plddt, pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
            shape_complementarity=d.shape_complementarity,
            boltz_affinity_score=d.boltz_affinity_score,   # RELATIVE RANK, never a K_D
            rosetta_dG=round(float(rdg), 2), solubility=0.3,
            contact_residues=",".join(d.contact_residues),
            cleft_overlap=pt.cleft_overlap(d.contact_residues, d.cleft),
            synthetic=d.synthetic,
        ))
    return pd.DataFrame(rows)

df_lin  = pool_to_df(linear); df_lin.to_csv("results/linear_designs.csv", index=False)
df_cyc  = pool_to_df(macro);  df_cyc.to_csv("results/macrocycle_designs.csv", index=False)
df_foil = pool_to_df(foil);   df_foil.to_csv("results/miniprotein_foil_designs.csv", index=False)
combined = pd.concat([df_lin, df_cyc, df_foil], ignore_index=True)
combined.to_csv("results/all_designs.csv", index=False)

print("wrote results/linear_designs.csv          ", df_lin.shape)
print("wrote results/macrocycle_designs.csv      ", df_cyc.shape)
print("wrote results/miniprotein_foil_designs.csv", df_foil.shape)
print("wrote results/all_designs.csv             ", combined.shape)
print("\nALL numbers are SYNTHETIC in the mock dry run (EXAMPLE_DATA). boltz_affinity_score is a")
print("RELATIVE RANK, never a K_D. Never report any of this as real results.")
combined.head(4)

## D2 checklist
- [ ] Linear pool generated across a **length** sweep (BoltzGen peptide-anything / EvoBind2 on Colab).
- [ ] Macrocycle pool generated across a **length + cyclization** sweep (macrocycle arm on Pro).
- [ ] Mini-protein **foil** generated (Project-06 workflow on A100) for the modality comparison.
- [ ] Every design scored by AF2/Boltz-2 (`pae_interaction` parsed; Boltz-2 affinity as a *rank*).
- [ ] Pools written to `results/`; design log (every config + seed + tool **commit/release** + path) in `LOG.md`.
- [ ] Version-verify output captured (incl. the BoltzGen-release check); 3–4 page interim report.

**Next:** `03_filter_and_rank.ipynb` — run the **shared** filter on the pools.

---

## ▶︎ Section 4 / 6 — `03_filter_and_rank.ipynb`

---

# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 09** you build `fp.Design` **binder** objects from each modality pool, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per modality** (linear / macrocycle / mini-protein foil) so the comparisons are fair (D3 part 1). The
**Boltz-2 affinity score rides along for ranking only — it is never a pass/fail and never a K_D.**

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/linear_designs.csv`, `results/macrocycle_designs.csv`, and
`results/miniprotein_foil_designs.csv` exist.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6. (The Boltz-2
affinity score is **not** a cutoff — it is a relative rank we keep alongside for prioritization.)

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

## Build `Design` (binder) objects from the modality pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and `rosetta_dG`/`shape_complementarity`/`solubility`
(Layer 3 physics). We keep `modality`, `cyclic`, `cleft_overlap`, and the Boltz-2 rank in `extra` for
the modality comparison + cleft analysis in notebook 04. (Mock has no independent orthogonal predictor,
so we run Layers 1+3 here; on Colab add a second predictor — e.g. AF2 ↔ Boltz-2 — for Layer 2.)

In [ ]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
NEEDED = ["results/linear_designs.csv", "results/macrocycle_designs.csv",
          "results/miniprotein_foil_designs.csv"]
if not all(os.path.exists(p) for p in NEEDED):
    import peptide_tools as pt
    TARGET, CLEFT = "MDM2", pt.parse_cleft("A54,A67,A73,A93,A100")
    def _mk(designs, p):
        rows = []
        for d in designs:
            rdg = -45.0 + (pt._hashints("dG", d.design_id) % 40)
            rows.append(dict(design_id=d.design_id, modality=d.modality, cyclic=d.cyclic,
                             length=d.length, sequence=d.sequence, plddt=d.plddt,
                             pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                             shape_complementarity=d.shape_complementarity,
                             boltz_affinity_score=d.boltz_affinity_score,
                             rosetta_dG=round(float(rdg), 2), solubility=0.3,
                             cleft_overlap=pt.cleft_overlap(d.contact_residues, d.cleft),
                             synthetic=d.synthetic))
        pd.DataFrame(rows).to_csv(p, index=False)
    lin = []
    for L in [8, 12, 16, 20]:
        lin += pt.design_peptide(TARGET, CLEFT, length=L, cyclic=False, n=15, tool="mock")
    cyc = []
    for L in [8, 11, 14]:
        cyc += pt.design_peptide(TARGET, CLEFT, length=L, cyclic=True, n=15, tool="mock")
    foil = pt.design_miniprotein_foil(TARGET, CLEFT, n=30, tool="mock")
    for g in (lin, cyc, foil):
        pt.score_designs(g, tool="mock")
    _mk(lin, NEEDED[0]); _mk(cyc, NEEDED[1]); _mk(foil, NEEDED[2])

df_lin  = pd.read_csv(NEEDED[0])
df_cyc  = pd.read_csv(NEEDED[1])
df_foil = pd.read_csv(NEEDED[2])

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"modality": r.get("modality"), "cyclic": bool(r.get("cyclic")),
               "cleft_overlap": r.get("cleft_overlap"),
               "boltz_affinity_score": r.get("boltz_affinity_score")},
    )

binders_lin  = [row_to_binder(r) for _, r in df_lin.iterrows()]
binders_cyc  = [row_to_binder(r) for _, r in df_cyc.iterrows()]
binders_foil = [row_to_binder(r) for _, r in df_foil.iterrows()]
print(f"built {len(binders_lin)} linear + {len(binders_cyc)} macrocycle + {len(binders_foil)} mini-protein Designs")

## Run the pipeline — per modality (fair comparison)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked
DataFrame with survival counts in `df.attrs`. We run **each modality separately** so the
survival-at-each-layer funnels are comparable. We use Layers 1+3 here (mock has no independent
orthogonal source; add Layer 2 on Colab with a second predictor).

In [ ]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["modality"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_lin  = run_one(binders_lin,  "linear")
ranked_cyc  = run_one(binders_cyc,  "macrocycle")
ranked_foil = run_one(binders_foil, "miniprotein")

ranked = pd.concat([ranked_lin, ranked_cyc, ranked_foil], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "modality", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **combined**
pool for one comparable figure; the per-modality runs above are the rigorous version. Read the bars as
a funnel: steep drops show which layer discriminates.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_lin + binders_cyc + binders_foil
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p09")
print("\nsaved results/p09_survival.png + results/p09_ranked.csv")
top

## Honest hit-rate accounting (per modality)

Report `N passing all layers / N generated` for **each** modality — this is the number the comparisons
in notebook 04 build on. Survival is *enrichment*, not *correctness*; and for short peptides predicted
affinity is unreliable (the Boltz-2 score is a rank, not a K_D). Mock numbers are SYNTHETIC.

In [ ]:
for label, df in [("linear", ranked_lin), ("macrocycle", ranked_cyc), ("miniprotein", ranked_foil)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per modality** (funnel figure `results/p09_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for linear, macrocycle, and mini-protein foil.
- [ ] Boltz-2 affinity kept as a **rank only** (never a cutoff, never a K_D); mapping assumptions written down.

**Next:** `04_validate.ipynb` — the linear-vs-cyclic + peptide-vs-protein comparisons.

---

## ▶︎ Section 5 / 6 — `04_validate.ipynb`

---

# 04 · Validate — linear-vs-cyclic + peptide-vs-protein modality comparison

**Standard slot:** *validate (in silico).* **For Project 09 this is the core comparison:** the two
modality benchmarks — **linear vs cyclic** (does cyclization help, at what cost?) and **peptide vs
mini-protein** (the foil) — plus **cleft-engagement reasoning vs p53** and **cyclization-feasibility
notes**, with publication-style figures (D3 part 2).

Needs `results/linear_designs.csv` + `results/macrocycle_designs.csv` +
`results/miniprotein_foil_designs.csv` + `results/all_ranked.csv` (from notebooks 02–03).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Head-to-head hit rate + interface metrics (per modality)

Compare the modalities on (a) all-layers **hit rate** and (b) the **pae_interaction** distribution of
survivors. A fair comparison filters all arms identically (notebook 03) and reports the *distribution*,
not the single best. The Boltz-2 affinity column is a **relative rank** we carry for prioritization,
never a K_D. Mock numbers are SYNTHETIC.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ranked = pd.read_csv("results/all_ranked.csv")
print("modalities:", ranked["modality"].value_counts().to_dict())

# Join the Boltz-2 rank + cleft_overlap from the pool CSVs onto the ranked survivors.
# (Drop any duplicate design_ids so the .map() index is unique — design_ids should already be unique.)
pools = pd.concat([pd.read_csv("results/linear_designs.csv"),
                   pd.read_csv("results/macrocycle_designs.csv"),
                   pd.read_csv("results/miniprotein_foil_designs.csv")], ignore_index=True)
pmap = pools.drop_duplicates("design_id").set_index("design_id")
ranked["boltz_affinity_score"] = ranked["design_id"].map(pmap["boltz_affinity_score"])
ranked["cleft_overlap"] = ranked["design_id"].map(pmap["cleft_overlap"])

summary = []
for m, g in ranked.groupby("modality"):
    n = len(g); passed = int((g["layers_passed"] >= 3).sum())
    summary.append(dict(modality=m, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_cleft_overlap=round(float(g["cleft_overlap"].median()), 3),
                        median_boltz_rank=round(float(g["boltz_affinity_score"].median()), 4)))
summary = pd.DataFrame(summary)
print("\nmodality summary (SYNTHETIC if mock; boltz rank is RELATIVE, not a K_D):")
print(summary.to_string(index=False))

In [ ]:
# pae_interaction + Boltz-2 relative-rank distributions per modality.
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for m, g in ranked.groupby("modality"):
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=m)
    ax[1].hist(g["boltz_affinity_score"].dropna(), bins=15, alpha=0.5, label=m)
ax[0].set_xlabel("pae_interaction (Å, lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2/Boltz-2 pae_interaction"); ax[0].legend()
ax[1].set_xlabel("Boltz-2 affinity score (RELATIVE rank, NOT a K_D)"); ax[1].set_title("Boltz-2 relative ranking"); ax[1].legend()
fig.suptitle("Modality comparison (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p09_modality.png", dpi=150); plt.show()
print("saved results/p09_modality.png")

## 2 · Linear vs cyclic, explicitly

The first headline comparison. Restrict to the two **peptide** modalities (drop the mini-protein foil)
and compare hit rate, `pae_interaction`, and cleft overlap. Frame the result with the chemistry:
cyclization typically **rigidifies** the bound conformation and buys **protease stability** — at a
**synthesis cost**. Higher predicted engagement for cyclic *and* a feasible ring is the win.

In [ ]:
lin_cyc = ranked[ranked["modality"].isin(["linear", "macrocycle"])]
print("LINEAR vs CYCLIC (survivors; SYNTHETIC if mock):")
for m, g in lin_cyc.groupby("modality"):
    surv = g[g["layers_passed"] >= 3]
    print(f"  {m:11s}: n={len(g)}  hit_rate={100*len(surv)/max(len(g),1):.1f}%  "
          f"median_pae={g['pae_interaction'].median():.2f}  "
          f"median_cleft_overlap={g['cleft_overlap'].median():.2f}")
print("\nInterpretation: cyclization rigidifies + buys protease stability (validate in nb05), at a")
print("synthesis cost. Report the DISTRIBUTION + N, not the single best. Predicted affinity is unreliable.")

## 3 · Peptide vs mini-protein (the foil), explicitly

The second headline comparison: when is a **peptide/macrocycle** the right modality versus a folded
**mini-binder**? Mini-proteins usually have **higher in-silico hit rates** and easy E. coli
expression; peptides/macrocycles offer **oral / cell-penetrant** potential but are harder to get right
and need SPPS + stability/permeability validation. Report both arms honestly.

In [ ]:
pep = ranked[ranked["modality"].isin(["linear", "macrocycle"])]
foil = ranked[ranked["modality"] == "miniprotein"]
def _hit(g):
    return 100 * int((g["layers_passed"] >= 3).sum()) / max(len(g), 1)
print("PEPTIDE/MACROCYCLE vs MINI-PROTEIN FOIL (SYNTHETIC if mock):")
print(f"  peptide+macrocycle: n={len(pep)}  hit_rate={_hit(pep):.1f}%  median_pae={pep['pae_interaction'].median():.2f}")
print(f"  mini-protein foil : n={len(foil)} hit_rate={_hit(foil):.1f}%  median_pae={foil['pae_interaction'].median():.2f}")
print("\nModality trade-off: mini-proteins = higher hit rate + easy expression; peptides/macrocycles =")
print("oral/cell-penetrant potential but modest hit rate + SPPS/stability/permeability validation.")

## 4 · Cleft-engagement vs p53 + cyclization-feasibility notes `[extension]`

A peptide only **displaces p53** if it covers the p53 sub-pockets. `cleft_overlap` is our geometry
proxy: the fraction of cleft residues the peptide contacts. Higher ⇒ more likely a competitor. We also
flag a **cyclization-feasibility** note per macrocycle (a teaching heuristic on ring size; on Colab,
reason about head-to-tail vs side-chain closure and non-canonical residues for real).

In [ ]:
surv = ranked[ranked["layers_passed"] >= 3].copy()
print("cleft-engagement overlap of survivors (SYNTHETIC if mock):")
for m, g in surv.groupby("modality"):
    print(f"  {m:11s}: median p53-cleft overlap = {g['cleft_overlap'].median():.2f}  (n={len(g)})")

# Likely p53 competitors = survivors that also cover enough of the cleft.
COMPETE_OVERLAP = 0.5
competitors = surv[surv["cleft_overlap"] >= COMPETE_OVERLAP]
print(f"\nlikely p53 competitors (survivor AND cleft_overlap>={COMPETE_OVERLAP}): {len(competitors)}")
print(competitors.groupby("modality").size().to_dict())

# Cyclization-feasibility heuristic (TEACHING ONLY): flag macrocycle ring sizes outside a sane window.
macro_pool = pd.read_csv("results/macrocycle_designs.csv")
def ring_feasible(L):
    # Head-to-tail macrocycles are typically synthesizable for ~5-15 residues; outside that, flag for review.
    return 5 <= int(L) <= 15
macro_pool["ring_feasible_heuristic"] = macro_pool["length"].apply(ring_feasible)
print("\nmacrocycle ring-size feasibility (TEACHING heuristic; verify real chemistry on Colab):")
print(macro_pool["ring_feasible_heuristic"].value_counts().to_dict())

## 5 · Select the top candidates per modality

The D★ deliverable wants the **top candidates per modality**. Rank survivors by the composite score
and, as a tie-breaker, prefer higher p53-cleft overlap (and, for macrocycles, a feasible ring). Save the
shortlist for the validation plan (notebook 05). The Boltz-2 rank can prioritize *synthesis order* —
not pass/fail.

In [ ]:
top_per = []
for m, g in ranked.groupby("modality"):
    g2 = g[g["layers_passed"] >= 3].sort_values(
        ["score", "cleft_overlap"], ascending=False).head(15)
    top_per.append(g2)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=15 per modality)")
print(top.groupby("modality").size().to_dict())
top.head(8)[["design_id", "modality", "score", "pae_interaction", "cleft_overlap", "boltz_affinity_score"]]

## D3 (part 2) checklist
- [ ] **Linear vs cyclic**: hit rate + pae + cleft overlap, framed with the stability/synthesis trade-off (figure `results/p09_modality.png`).
- [ ] **Peptide vs mini-protein**: the foil comparison, honest about both arms' trade-offs.
- [ ] Cleft-engagement vs p53: overlap of survivors; "likely p53 competitor" count.
- [ ] Cyclization-feasibility notes (ring size; head-to-tail vs side-chain) for macrocycles.
- [ ] `results/top_candidates.csv`: top candidates per modality, ready for the validation plan.
- [ ] Honest discussion of failure modes + "predicted affinity is unreliable for short peptides".

**Next:** `05_validation_plan.ipynb` — the SPPS + protease-stability + permeability plan.

---

## ▶︎ Section 6 / 6 — `05_validation_plan.ipynb`

---

# 05 · Validation Plan — SPPS + protease stability + permeability + controls

**Standard slot:** *validation plan.* **For Project 09 this means:** turn the top candidates into a
**costed, controlled, peptide-appropriate** wet-lab plan. Peptides are **NOT** validated like E. coli
mini-binders — they are made by **solid-phase peptide synthesis (SPPS)** and need **protease-stability**
and **permeability** assays, plus a binding assay and the mandatory controls (positive: a known
p53-peptide; **scrambled-sequence** negative; unrelated negative). Includes the **D-amino-acid /
stapling** stretch and the **Boltz-2 affinity** ranking scaffold (D4/D5).

A design that passes every filter is a **hypothesis** — synthesis + binding/stability/permeability
assays are what test it, and predicted affinity is unreliable. Needs `results/top_candidates.csv`
(notebook 04).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Draft the peptide validation plan

Generate a plan card from the top candidates: synthesis (SPPS), assays (binding + **stability** +
**permeability**), controls, timeline, costed reagents. Fill the `<...>` from your own numbers; this is
the deliverable other people will actually read.

In [ ]:
import pandas as pd, os

top = pd.read_csv("results/top_candidates.csv") if os.path.exists("results/top_candidates.csv") else pd.DataFrame()
n_top = len(top)
by_mod = top.groupby("modality").size().to_dict() if n_top else {}

plan = f"""# MDM2 Peptide / Macrocycle Validation Plan (Project 09 — by <your name>, <date>)

## Candidates
Top {n_top} candidates carried forward ({by_mod}); see results/top_candidates.csv.
EVERY in-silico number is a HYPOTHESIS until measured. `pae_interaction` is confidence, not affinity;
the Boltz-2 affinity score is a RELATIVE RANK, not a K_D. Predicted affinity for short peptides is unreliable.

## Synthesis strategy (peptides are NOT made in E. coli)
- Linear peptides: solid-phase peptide synthesis (SPPS, Fmoc); HPLC purify; confirm by mass spec.
- Macrocycles: SPPS + cyclization (head-to-tail lactam / side-chain / disulfide, or a hydrocarbon staple).
  D-amino acids / N-methylation introduced here (raises synthesis complexity — see the stretch below).

## Assays (binding -> stability -> permeability -> function)
1. Binding: SPR/BLI vs immobilized MDM2, OR a fluorescence-polarization DISPLACEMENT assay of a labeled
   p53 peptide (does the design displace p53 from MDM2?). Test a dilution series.
2. Protease stability (the key reason to cyclize): serum / trypsin / chymotrypsin half-life, linear vs cyclic.
3. Permeability (the key reason a macrocycle could be oral/cell-penetrant): PAMPA and/or Caco-2.
4. Function (the point): cell-based p53-pathway reactivation (e.g. p53-reporter / p21 induction) in MDM2-amplified cells.

## Controls (MANDATORY)
- Positive: a KNOWN p53-mimetic / stapled peptide (e.g. the ATSP-7041 lineage) -> assay + MDM2 reagent are active.
- Negative (scrambled-sequence): YOUR OWN top design with its sequence scrambled -> must LOSE binding (cleanest specificity control).
- Negative (unrelated): an unrelated peptide of similar length -> should not bind MDM2.

## Realistic expectations
De novo peptide/macrocycle hit rates are MODEST and chemistry-dependent; predicted affinity is
UNRELIABLE for short peptides. Expect to synthesize many to find a few real, stable, permeable binders.
Report the experimental hit rate honestly. Do NOT imply a working peptide or fabricate a K_D.

## Timeline + costed reagents (fill in)
- SPPS synthesis ({n_top} peptides + scrambled-sequence negatives; macrocyclization adds steps): ${'{'}'<...>'{'}'}, <...> weeks.
- MDM2 reagent + labeled p53 peptide (FP) or SPR/BLI chips + positive-control peptide: $<...>.
- Protease-stability + PAMPA/Caco-2 assays: $<...>.
- Personnel / instrument time: <...> weeks.

## Responsible research
Competitive p53-mimetic peptides to a human oncology PPI (MDM2) to restore p53 function (in scope).
Synthesis via a biosecurity-screening provider; wet lab under institutional biosafety/ethics approval.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill the <...> placeholders from your numbers.")
print(plan[:700], "...")

## 2 · Build the scrambled-sequence negative controls

The single cleanest specificity control: take each top design and **scramble its sequence** — it should
**lose** binding. Synthesizing these alongside the real designs (same SPPS batch) makes the binding
comparison airtight. Here we scaffold the scramble deterministically.

In [ ]:
import random
import peptide_tools as pt   # pt._hashints gives a DETERMINISTIC seed (Python's hash() is salted)

def scramble_sequence(seq, seed=0):
    """Deterministically shuffle the whole peptide as a NEGATIVE-CONTROL stand-in.
    A scrambled p53-mimetic should lose the Phe/Trp/Leu register that engages the MDM2 cleft."""
    rng = random.Random(seed)
    s = list(seq)
    rng.shuffle(s)
    return "".join(s)

negs = []
if n_top and "sequence" in top.columns:
    # sequences live in the pool CSVs; join them onto the top candidates.
    pools = pd.concat([pd.read_csv("results/linear_designs.csv"),
                       pd.read_csv("results/macrocycle_designs.csv"),
                       pd.read_csv("results/miniprotein_foil_designs.csv")], ignore_index=True)
    seqmap = pools.drop_duplicates("design_id").set_index("design_id")["sequence"]
    for _, r in top.iterrows():
        s = str(seqmap.get(r["design_id"], r.get("sequence", "")))
        if s and s != "nan":
            negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                             parent=r["design_id"], modality=r.get("modality"),
                             sequence=scramble_sequence(s, seed=pt._hashints(r["design_id"]) % 10**6),
                             role="scrambled-sequence negative control"))
    pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
    print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-sequence negatives")
else:
    # The top_candidates table may not carry sequences; pull them from the pools regardless.
    if n_top:
        pools = pd.concat([pd.read_csv("results/linear_designs.csv"),
                           pd.read_csv("results/macrocycle_designs.csv"),
                           pd.read_csv("results/miniprotein_foil_designs.csv")], ignore_index=True)
        seqmap = pools.drop_duplicates("design_id").set_index("design_id")["sequence"]
        for _, r in top.iterrows():
            s = str(seqmap.get(r["design_id"], ""))
            if s and s != "nan":
                negs.append(dict(design_id=str(r["design_id"]) + "_SCRAM",
                                 parent=r["design_id"], modality=r.get("modality"),
                                 sequence=scramble_sequence(s, seed=pt._hashints(r["design_id"]) % 10**6),
                                 role="scrambled-sequence negative control"))
        pd.DataFrame(negs).to_csv("results/negative_controls.csv", index=False)
        print(f"wrote results/negative_controls.csv: {len(negs)} scrambled-sequence negatives")
    else:
        print("Run notebook 04 first to produce results/top_candidates.csv.")

## 3 · (Stretch) D-amino-acid / stapling extension `[stretch]`

Cyclization is not the only stability/permeability lever. **D-amino-acid substitutions**, **N-methylation**,
and **hydrocarbon stapling** further resist proteases and can improve permeability — at a real synthesis
cost. Propose specific modifications for your top macrocycles and reason about the trade-off. This is a
*design proposal*, not a prediction — do not claim a stability/permeability number you have not measured.

In [ ]:
# Scaffold ONLY — a design proposal, not a measurement.
# For each top macrocycle, propose (and justify) modifications, then weigh synthesis cost vs benefit:
#   - D-amino-acid substitution at protease-cleavage-prone positions (e.g. after Arg/Lys);
#   - N-methylation of backbone amides to block proteolysis + aid membrane permeability;
#   - a hydrocarbon staple (i, i+4 / i, i+7) to lock the bound helix-mimetic conformation.
# Report these as HYPOTHESES; protease half-life + PAMPA/Caco-2 in the wet-lab plan TEST them.
print("D-amino-acid / N-methylation / stapling = stretch DESIGN proposals (stability/permeability levers).")
print("Weigh synthesis complexity vs the protease-stability / permeability benefit. Measure, never assume.")

## 4 · (Stretch) Boltz-2 affinity to prioritize synthesis order `[stretch]`

Boltz-2 can predict an affinity *signal* for the top complexes. Use it for **relative ranking +
caveats only** — to decide **which hits to synthesize first** — **never** as evidence of binding and
**never** as a K_D. (You already carried `boltz_affinity_score` through the pipeline; here it only sets
priority.)

In [ ]:
import pandas as pd, os
if os.path.exists("results/top_candidates.csv"):
    top = pd.read_csv("results/top_candidates.csv")
    if "boltz_affinity_score" in top.columns and top["boltz_affinity_score"].notna().any():
        order = top.sort_values("boltz_affinity_score", ascending=False)
        print("Suggested SYNTHESIS ORDER by Boltz-2 RELATIVE rank (NOT a K_D; SYNTHETIC if mock):")
        print(order[["design_id", "modality", "boltz_affinity_score"]].head(10).to_string(index=False))
    else:
        print("No Boltz-2 rank present — populate boltz_affinity_score on Colab (relative rank only).")
print("\nReminder: Boltz-2 affinity = a RELATIVE prioritization signal. NEVER a fabricated K_D (MASTER_BLUEPRINT §9).")

## D4 / D5 checklist
- [ ] `results/validation_plan.md` completed: **SPPS** synthesis + binding + **protease-stability** + **permeability** + function, timeline, costed reagents.
- [ ] Controls specified: positive (known p53-peptide / stapled peptide), **scrambled-sequence** negative (`results/negative_controls.csv`), unrelated negative.
- [ ] (Stretch) D-amino-acid / N-methylation / stapling proposals for top macrocycles, with the synthesis-cost trade-off.
- [ ] (Stretch) Boltz-2 affinity used only to set synthesis priority — relative rank, no fabricated K_D.
- [ ] Honest framing: every design is a hypothesis until synthesis + assays; report the experimental hit rate.
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a peptide/macrocycle capstone that mirrors the binder-family workflow with peptide
chemistry, SPPS validation, and the linear-vs-cyclic + peptide-vs-protein modality study at its core.